# Лабораторная работа №8: Active learning и human-in-the-loop

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Реализовать random, entropy и diversity sampling.
- Провести несколько AL-итераций.
- Построить learning curves при равном бюджете.


## Теоретическая часть

Active learning выбирает следующий batch из pool. Uncertainty ищет неуверенные объекты, diversity покрывает пространство, disagreement использует расхождение моделей. Сравнение корректно только при одинаковом annotation budget.


### Математическая постановка

Entropy sampling: $u(x)=-\sum_c p(c\mid x)\log p(c\mid x)$. Diversity можно реализовать k-center: выбрать объект, максимизирующий расстояние до уже размеченного множества.


### Материалы

- [Active Learning survey](https://arxiv.org/abs/2009.00236)
- [scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Работа выполняется на изображениях рукописных цифр `sklearn.datasets.load_digits`; oracle в этом воспроизводимом эксперименте симулирует человека, открывая истинную метку только после отбора объекта. FAST_DEV_RUN — два раунда проверки кода, полный режим — восемь раундов и повторный random baseline.

In [1]:
# ===== Настройка и конфиг =====
from pathlib import Path
import json, random
import numpy as np

SEED = 42
FAST_DEV_RUN = False
RUN_BONUS = True

ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

if FAST_DEV_RUN:
    N_ROUNDS, BATCH, N_PER_CLASS_INIT, N_RANDOM_REPEATS = 2, 5, 1, 2
else:
    N_ROUNDS, BATCH, N_PER_CLASS_INIT, N_RANDOM_REPEATS = 8, 10, 2, 5

def save_json(name, data):
    (ARTIFACTS / name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print(f"Конфиг: rounds={N_ROUNDS}, batch={BATCH}, initial={N_PER_CLASS_INIT}/класс, random_repeats={N_RANDOM_REPEATS}")

Конфиг: rounds=8, batch=10, initial=2/класс, random_repeats=5


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Стратифицированный отложенный тест

*Вес: 1 балл.*

**Постановка.** Этап «стратифицированный отложенный тест» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Разделите изображения цифр по стратифицированному правилу до выбора стратегии.
2. Сохраните и предъявите индексы train/test изображений цифр.

**Результат.** индексы train/test изображений цифр.

**Критерий принятия.** Задание считается принятым, если индексы train/test изображений цифр получены из фактических входов этого этапа и доступны для проверки.

In [2]:
# ===== Этап 1. Стратифицированный train/test (до всякого выбора стратегий) =====
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X, y, images = digits.data, digits.target, digits.images

idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.3, stratify=y, random_state=SEED)

def class_counts(idx):
    return {int(c): int((y[idx] == c).sum()) for c in range(10)}

save_json("split_indices.json", {
    "n_train": len(idx_train), "n_test": len(idx_test),
    "train_indices": idx_train.tolist(), "test_indices": idx_test.tolist(),
    "train_class_counts": class_counts(idx_train), "test_class_counts": class_counts(idx_test),
    "seed": SEED, "rule": "train_test_split(stratify=y, test_size=0.3)",
})
print(f"train={len(idx_train)}, test={len(idx_test)}")
print("test по классам:", class_counts(idx_test))
print("-> artifacts/split_indices.json")

train=1257, test=540
test по классам: {0: 54, 1: 55, 2: 53, 3: 55, 4: 54, 5: 55, 6: 54, 7: 54, 8: 52, 9: 54}
-> artifacts/split_indices.json


### Начальная размеченная выборка

*Вес: 1 балл.*

**Постановка.** Этап «начальная размеченная выборка» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Задайте initial с каждым классом и неразмеченный pool только из train.
2. Сохраните и предъявите непересекающиеся initial и pool.

**Результат.** непересекающиеся initial и pool.

**Критерий принятия.** Задание считается принятым, если непересекающиеся initial и pool получены из фактических входов этого этапа и доступны для проверки.

In [3]:
# ===== Этап 2. Initial (каждый класс) + pool только из train =====
rng = np.random.default_rng(SEED + 100)

initial = []
for c in range(10):
    cand = idx_train[y[idx_train] == c]
    initial += rng.choice(cand, size=N_PER_CLASS_INIT, replace=False).tolist()
initial = np.array(sorted(initial))
pool = np.array(sorted(set(idx_train.tolist()) - set(initial.tolist())))

assert len(set(initial.tolist()) & set(pool.tolist())) == 0, "initial и pool пересекаются!"
assert set(initial.tolist()) | set(pool.tolist()) == set(idx_train.tolist()), "initial+pool != train"

save_json("initial_pool.json", {
    "initial_indices": initial.tolist(), "pool_indices": pool.tolist(),
    "n_initial": len(initial), "n_pool": len(pool),
    "initial_class_counts": {int(c): int((y[initial] == c).sum()) for c in range(10)},
    "note": "initial покрывает каждый класс; pool = train \\ initial; test не входит ни в один из них",
})
print(f"initial={len(initial)} (по {N_PER_CLASS_INIT} на класс), pool={len(pool)}, пересечение=0")
print("-> artifacts/initial_pool.json")

initial=20 (по 2 на класс), pool=1237, пересечение=0
-> artifacts/initial_pool.json


### Протокол симулированного разметчика

*Вес: 1 балл.*

**Постановка.** Этап «протокол симулированного разметчика» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните split и документируйте, что oracle открывает метку лишь после выбора.
2. Сохраните и предъявите artifacts/split.json с происхождением меток.

**Результат.** artifacts/split.json с происхождением меток.

**Критерий принятия.** Задание считается принятым, если artifacts/split.json с происхождением меток получены из фактических входов этого этапа и доступны для проверки.

In [4]:
# ===== Этап 3. Протокол оракула -> artifacts/split.json =====
save_json("split.json", {
    "dataset": "sklearn.datasets.load_digits (1797 изображений 8x8, 10 классов)",
    "split_rule": "train_test_split(test_size=0.3, stratify=y, random_state=42)",
    "n_train": len(idx_train), "n_test": len(idx_test),
    "train_indices": idx_train.tolist(), "test_indices": idx_test.tolist(),
    "initial_labeled_indices": initial.tolist(),
    "pool_unlabeled_indices": pool.tolist(),
    "oracle_protocol": {
        "who": "oracle симулирует человека-разметчика",
        "rule": "истинная метка y[i] открывается ТОЛЬКО ПОСЛЕ того, как стратегия выбрала объект i",
        "initial": "метки initial открыты оракулом на старте (отбор по классам — стандартный stratified seeding)",
        "pool": "метки pool скрыты от стратегий; функции выбора получают только индексы и признаки X, без y",
        "test": "метки test никогда не участвуют в обучении — только для оценки accuracy",
    },
    "label_provenance": {
        "initial": "oracle (старт, до раундов)",
        "queried": "oracle (после выбора, фиксируется в artifacts/traces.json: opened_labels)",
        "pool_rest": "не открыты",
    },
})
print("-> artifacts/split.json (split + происхождение меток)")

-> artifacts/split.json (split + происхождение меток)


### Три стратегии запроса

*Вес: 1 балл.*

**Постановка.** Этап «три стратегии запроса» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Реализуйте случайный, энтропийный и k-center выбор; в k-center обновляйте расстояния после каждого объекта.
2. Сохраните и предъявите выбор random, entropy и k-center.

**Результат.** выбор random, entropy и k-center.

**Критерий принятия.** Задание считается принятым, если выбор random, entropy и k-center получены из фактических входов этого этапа и доступны для проверки.

In [5]:
# ===== Этап 4. Стратегии: random / entropy / k-center =====
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances

def fit_model(labeled_idx, labels):
    return LogisticRegression(max_iter=2000).fit(X[labeled_idx], labels)

def select_random(pool_idx, labeled_idx, labels, k, rng):
    cand = np.asarray(pool_idx); k = min(k, len(cand))
    return rng.choice(cand, size=k, replace=False).tolist()

def select_entropy(pool_idx, labeled_idx, labels, k, rng):
    cand = np.asarray(pool_idx); k = min(k, len(cand))
    P = fit_model(labeled_idx, labels).predict_proba(X[cand])
    ent = -(P * np.log(np.clip(P, 1e-12, None))).sum(axis=1)     # u(x) = -sum p log p
    return cand[np.argsort(-ent)[:k]].tolist()

def select_kcenter(pool_idx, labeled_idx, labels, k, rng):
    cand = np.asarray(pool_idx); k = min(k, len(cand))
    P = X[cand]
    min_d = pairwise_distances(P, X[labeled_idx]).min(axis=1)    # дистанция до ближайшего размеченного
    chosen = []
    for _ in range(k):
        j = int(np.argmax(min_d))                                # самый удалённый объект
        chosen.append(j)
        # k-center: расстояния обновляются ПОСЛЕ каждого выбранного объекта
        min_d = np.minimum(min_d, np.sqrt(((P - P[j]) ** 2).sum(axis=1)))
        min_d[j] = -1.0
    return cand[chosen].tolist()

STRATEGIES = {"random": select_random, "entropy": select_entropy, "kcenter": select_kcenter}

# демонстрация: по одному батчу каждой стратегии из одинакового состояния (метки pool не видны)
demo = {}
for name, fn in STRATEGIES.items():
    sel = fn(pool, initial, y[initial], k=5, rng=np.random.default_rng(SEED + 200))
    demo[name] = [int(i) for i in sel]
    print(f"{name:8s} -> {demo[name]}  (классы: {[int(y[i]) for i in sel]})")
save_json("strategies_demo.json", {"batch_size": 5, "selections": demo})
print("-> artifacts/strategies_demo.json")

random   -> [604, 152, 1315, 1561, 784]  (классы: [6, 2, 8, 6, 8])
entropy  -> [555, 785, 621, 794, 914]  (классы: [9, 9, 9, 8, 9])
kcenter  -> [1277, 1595, 926, 889, 1024]  (классы: [5, 7, 1, 2, 9])
-> artifacts/strategies_demo.json


### Итерационное переобучение

*Вес: 1 балл.*

**Постановка.** Этап «итерационное переобучение» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. После каждого запроса дообучайте классификатор на открытых метках и фиксируйте trace.
2. Сохраните и предъявите функция run с фактическими запрошенными метками.

**Результат.** функция run с фактическими запрошенными метками.

**Критерий принятия.** Задание считается принятым, если функция run с фактическими запрошенными метками получены из фактических входов этого этапа и доступны для проверки.

In [6]:
# ===== Этап 5. run(): запрос -> oracle открывает метку -> дообучение -> trace =====
from sklearn.metrics import accuracy_score

def run(strategy_name, seed, registry=None):
    """Один AL-прогон. Стратегия выбирает объекты БЕЗ меток pool; oracle открывает метки выбранным."""
    fn = (registry or STRATEGIES)[strategy_name]
    rng = np.random.default_rng(seed)
    labeled, labels = list(initial.tolist()), list(y[initial].tolist())
    pool_now = pool.tolist()
    clf = fit_model(np.array(labeled), np.array(labels))
    curve = [{"budget": len(labeled),
              "test_acc": round(float(accuracy_score(y[idx_test], clf.predict(X[idx_test]))), 4)}]
    rounds = []
    for r in range(N_ROUNDS):
        asked = fn(pool_now, labeled, labels, BATCH, rng)      # выбор без меток
        opened = [int(y[i]) for i in asked]                     # oracle открывает метки
        labeled += [int(i) for i in asked]; labels += opened
        pool_now = [i for i in pool_now if i not in set(asked)]
        clf = fit_model(np.array(labeled), np.array(labels))    # переобучение после каждого запроса
        curve.append({"budget": len(labeled),
                      "test_acc": round(float(accuracy_score(y[idx_test], clf.predict(X[idx_test]))), 4)})
        rounds.append({"round": r + 1, "asked_indices": [int(i) for i in asked], "opened_labels": opened})
    return {"strategy": strategy_name, "seed": seed, "curve": curve, "rounds": rounds,
            "final_budget": len(labeled), "final_acc": curve[-1]["test_acc"]}

traces = {name: run(name, SEED) for name in STRATEGIES}
for name, tr in traces.items():
    print(f"{name:8s} " + " -> ".join(f"{p['budget']}:{p['test_acc']:.2f}" for p in tr["curve"]))
save_json("traces.json", traces)
print("-> artifacts/traces.json (запрошенные индексы + открытые оракулом метки)")

random   20:0.74 -> 30:0.81 -> 40:0.88 -> 50:0.89 -> 60:0.90 -> 70:0.91 -> 80:0.91 -> 90:0.92 -> 100:0.91
entropy  20:0.74 -> 30:0.75 -> 40:0.79 -> 50:0.88 -> 60:0.86 -> 70:0.88 -> 80:0.90 -> 90:0.89 -> 100:0.90
kcenter  20:0.74 -> 30:0.81 -> 40:0.80 -> 50:0.79 -> 60:0.85 -> 70:0.86 -> 80:0.86 -> 90:0.90 -> 100:0.91
-> artifacts/traces.json (запрошенные индексы + открытые оракулом метки)


### Кривые при общем бюджете

*Вес: 1 балл.*

**Постановка.** Этап «кривые при общем бюджете» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вычислите точки кривых для одинакового числа запросов и сохраните бюджеты.
2. Сохраните и предъявите artifacts/learning_curves.json.

**Результат.** artifacts/learning_curves.json.

**Критерий принятия.** Задание считается принятым, если artifacts/learning_curves.json получены из фактических входов этого этапа и доступны для проверки.

In [7]:
# ===== Этап 6. Learning curves при равном бюджете -> artifacts/learning_curves.json =====
budgets = [p["budget"] for p in traces["random"]["curve"]]
assert all([p["budget"] for p in traces[n]["curve"]] == budgets for n in STRATEGIES), "бюджеты разъехались"

save_json("learning_curves.json", {
    "protocol": {"n_initial": len(initial), "batch": BATCH, "n_rounds": N_ROUNDS,
                 "note": "все стратегии стартуют с одинакового initial и делают одинаковые запросы"},
    "budgets": budgets,
    "curves": {n: [{"budget": p["budget"], "test_acc": p["test_acc"]} for p in traces[n]["curve"]]
               for n in STRATEGIES},
})
print("Бюджеты:", budgets)
print("-> artifacts/learning_curves.json")

Бюджеты: [20, 30, 40, 50, 60, 70, 80, 90, 100]
-> artifacts/learning_curves.json


### Интеграл по бюджету разметки

*Вес: 1 балл.*

**Постановка.** Этап «интеграл по бюджету разметки» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Проинтегрируйте кривые accuracy по числу полученных меток.
2. Сохраните и предъявите AUC и конечная accuracy из фактических кривых.

**Результат.** AUC и конечная accuracy из фактических кривых.

**Критерий принятия.** Задание считается принятым, если AUC и конечная accuracy из фактических кривых получены из фактических входов этого этапа и доступны для проверки.

In [8]:
# ===== Этап 7. AUC: интеграл accuracy по числу открытых меток =====
trapz = getattr(np, "trapezoid", None) or np.trapz

def budget_auc(curve):
    b = np.array([p["budget"] for p in curve], dtype=float)
    a = np.array([p["test_acc"] for p in curve], dtype=float)
    return float(trapz(a, b) / (b[-1] - b[0]))    # средняя accuracy на единицу бюджета

auc_table = {n: round(budget_auc(traces[n]["curve"]), 4) for n in STRATEGIES}
final_acc = {n: traces[n]["final_acc"] for n in STRATEGIES}
save_json("auc_results.json", {
    "auc_by_budget": auc_table, "final_accuracy": final_acc,
    "definition": "AUC = интеграл accuracy по числу открытых меток, нормированный на длину бюджета",
})
for n in sorted(auc_table, key=auc_table.get, reverse=True):
    print(f"{n:8s} AUC={auc_table[n]:.4f}  final_acc={final_acc[n]:.3f}")
print("-> artifacts/auc_results.json")

random   AUC=0.8795  final_acc=0.913
entropy  AUC=0.8448  final_acc=0.898
kcenter  AUC=0.8376  final_acc=0.906
-> artifacts/auc_results.json


### Повторные случайные запросы

*Вес: 1 балл.*

**Постановка.** Этап «повторные случайные запросы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Повторите random с разными генераторами при неизменном train/test.
2. Сохраните и предъявите результаты random при различных seed.

**Результат.** результаты random при различных seed.

**Критерий принятия.** Задание считается принятым, если результаты random при различных seed получены из фактических входов этого этапа и доступны для проверки.

In [9]:
# ===== Этап 8. Random при разных генераторах (train/test и initial неизменны) =====
repeats = {}
for s in range(N_RANDOM_REPEATS):
    seed_r = SEED + 1000 + s
    tr = run("random", seed_r)
    repeats[f"seed_{seed_r}"] = {"auc": round(budget_auc(tr["curve"]), 4), "final_acc": tr["final_acc"]}
    print(f"random seed={seed_r}: AUC={repeats[f'seed_{seed_r}']['auc']:.4f}, final={tr['final_acc']:.3f}")

accs = np.array([v["final_acc"] for v in repeats.values()])
aucs = np.array([v["auc"] for v in repeats.values()])
save_json("random_repeats.json", {
    "repeats": repeats, "n_repeats": len(repeats),
    "final_acc_mean_std": [round(float(accs.mean()), 4), round(float(accs.std()), 4)],
    "auc_mean_std": [round(float(aucs.mean()), 4), round(float(aucs.std()), 4)],
    "note": "train/test и initial зафиксированы; меняется только генератор выбора",
})
print(f"random: final_acc {accs.mean():.3f}±{accs.std():.3f}, AUC {aucs.mean():.4f}±{aucs.std():.4f}")
print("-> artifacts/random_repeats.json")

random seed=1042: AUC=0.8324, final=0.893
random seed=1043: AUC=0.7963, final=0.889
random seed=1044: AUC=0.8492, final=0.913
random seed=1045: AUC=0.8316, final=0.917
random seed=1046: AUC=0.8669, final=0.922
random: final_acc 0.907±0.013, AUC 0.8353±0.0234
-> artifacts/random_repeats.json


### Разбор ошибок отложенного теста

*Вес: 1 балл.*

**Постановка.** Этап «разбор ошибок отложенного теста» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните изображения и истинные/предсказанные классы ошибочных test-примеров.
2. Сохраните и предъявите artifacts/test_errors.json с неверно распознанными изображениями.

**Результат.** artifacts/test_errors.json с неверно распознанными изображениями.

**Критерий принятия.** Задание считается принятым, если artifacts/test_errors.json с неверно распознанными изображениями получены из фактических входов этого этапа и доступны для проверки.

In [10]:
# ===== Этап 9. Ошибки финальной модели лучшей стратегии -> artifacts/test_errors.json =====
from collections import Counter

best_name = max(auc_table, key=auc_table.get)
tr_best = traces[best_name]
opened = [i for r in tr_best["rounds"] for i in r["asked_indices"]]
labeled_final = initial.tolist() + opened
clf_final = fit_model(np.array(labeled_final), np.array(y[labeled_final]))

pred = clf_final.predict(X[idx_test])
yt = y[idx_test]
wrong = np.where(pred != yt)[0]
errors = [{"test_index": int(idx_test[j]), "true": int(yt[j]), "pred": int(pred[j]),
           "image_8x8": images[idx_test[j]].astype(int).tolist()} for j in wrong]
pairs = dict(Counter(f"{int(yt[j])}->{int(pred[j])}" for j in wrong))

save_json("test_errors.json", {
    "strategy": best_name, "final_budget": len(labeled_final),
    "n_test": len(idx_test), "n_errors": int(len(wrong)),
    "error_rate": round(float(len(wrong) / len(idx_test)), 4),
    "confusion_pairs": pairs, "errors": errors,
})
print(f"Ошибок: {len(wrong)}/{len(idx_test)} ({len(wrong)/len(idx_test):.1%}); "
      f"top пары: {sorted(pairs.items(), key=lambda kv: -kv[1])[:5]}")
print("-> artifacts/test_errors.json (изображения + true/pred)")

Ошибок: 47/540 (8.7%); top пары: [('8->1', 8), ('7->1', 5), ('1->9', 4), ('9->7', 4), ('2->1', 4)]
-> artifacts/test_errors.json (изображения + true/pred)


### Сводка сравнительного эксперимента

*Вес: 1 балл.*

**Постановка.** Этап «сводка сравнительного эксперимента» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сопоставьте AUC и конечную точность по трём стратегиям.
2. Сохраните и предъявите artifacts/comparison.json.

**Результат.** artifacts/comparison.json.

**Критерий принятия.** Задание считается принятым, если artifacts/comparison.json получены из фактических входов этого этапа и доступны для проверки.

In [11]:
# ===== Этап 10. Сводка по трём стратегиям -> artifacts/comparison.json =====
rep = json.loads((ARTIFACTS / "random_repeats.json").read_text(encoding="utf-8"))
ranking = sorted(STRATEGIES, key=lambda n: auc_table[n], reverse=True)
comparison = {
    "protocol": {"n_initial": len(initial), "batch": BATCH, "n_rounds": N_ROUNDS,
                 "final_budget": traces["random"]["final_budget"], "n_test": len(idx_test),
                 "classifier": "LogisticRegression(max_iter=2000) на сырых пикселях 8x8"},
    "strategies": {n: {"auc": auc_table[n], "final_acc": final_acc[n]} for n in STRATEGIES},
    "random_repeats": {"n": rep["n_repeats"], "auc_mean_std": rep["auc_mean_std"],
                       "final_acc_mean_std": rep["final_acc_mean_std"]},
    "ranking_by_auc": ranking,
    "conclusion": (f"Лучшая стратегия по AUC: {ranking[0]}. Разрыв с random корректен только с учётом "
                   f"разброса повторов (random final_acc {rep['final_acc_mean_std'][0]:.3f}"
                   f"±{rep['final_acc_mean_std'][1]:.3f})."),
}
save_json("comparison.json", comparison)
print(json.dumps(comparison["strategies"], ensure_ascii=False, indent=2))
print("Рейтинг по AUC:", ranking)
print("-> artifacts/comparison.json")

{
  "random": {
    "auc": 0.8795,
    "final_acc": 0.913
  },
  "entropy": {
    "auc": 0.8448,
    "final_acc": 0.8981
  },
  "kcenter": {
    "auc": 0.8376,
    "final_acc": 0.9056
  }
}
Рейтинг по AUC: ['random', 'entropy', 'kcenter']
-> artifacts/comparison.json


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [12]:
# ===== Бонус: батч-разнообразие uncertainty-выборки =====
# Гипотеза (сформулирована ДО проверки): чистый entropy набирает батч из похожих объектов
# одного предсказанного класса — польза меток дублируется. Лимит "не более 1 объекта на
# предсказанный класс за раунд" должен поднять AUC без изменения модели и бюджета.
from collections import Counter as _C

def select_entropy_diverse(pool_idx, labeled_idx, labels, k, rng):
    cand = np.asarray(pool_idx); k = min(k, len(cand))
    clf = fit_model(labeled_idx, labels)
    P = clf.predict_proba(X[cand])
    ent = -(P * np.log(np.clip(P, 1e-12, None))).sum(axis=1)
    pred_cls = P.argmax(axis=1)
    cap = max(1, k // 10)
    picked, cnt = [], {}
    for j in np.argsort(-ent):
        c = int(pred_cls[j])
        if cnt.get(c, 0) < cap:
            picked.append(int(cand[j])); cnt[c] = cnt.get(c, 0) + 1
        if len(picked) == k:
            break
    return picked

if RUN_BONUS:
    reg = dict(STRATEGIES); reg["entropy_diverse"] = select_entropy_diverse
    div_runs = [run("entropy_diverse", s, reg) for s in (SEED, SEED + 1001, SEED + 1002)]
    auc_div = [round(budget_auc(t["curve"]), 4) for t in div_runs]
    rows = {"random": {"auc": auc_table["random"], "final_acc": final_acc["random"]},
            "entropy": {"auc": auc_table["entropy"], "final_acc": final_acc["entropy"]},
            "entropy_diverse": {"auc": round(float(np.mean(auc_div)), 4),
                                 "final_acc": round(float(np.mean([t["final_acc"] for t in div_runs])), 4)}}
    imp = rows["entropy_diverse"]["auc"] - rows["entropy"]["auc"]
    expl = ("entropy действительно сгущает батч в 1-2 кластерах; лимит по классу добавил разнообразие"
            if imp > 0 else
            "на хорошо разделимом digits чистый entropy уже покрывает разные классы; лимит не даёт выигрыша")
    save_json("bonus.json", {"hypothesis": "лимит по предсказанному классу в entropy-батче увеличивает AUC",
                             "protocol": "тот же initial/test/бюджет; entropy_diverse, 3 сида", "results": rows,
                             "auc_diverse_by_seed": auc_div, "delta_auc_vs_entropy": round(float(imp), 4)})
    (ARTIFACTS / "bonus.md").write_text(
        "# Бонусное мини-исследование: батч-разнообразие uncertainty-запроса\n\n"
        "**Гипотеза.** Чистый entropy-запрос собирает батч из похожих объектов одного предсказанного "
        "класса; польза их меток дублируется. Ограничение не более 1 объекта на предсказанный класс "
        "в батче должно увеличить AUC.\n\n"
        f"**Протокол.** load_digits, тот же initial/test/бюджет, что в основном эксперименте; сравнение "
        f"random / entropy / entropy_diverse (лимит по предсказанному классу); 3 сида для diverse; "
        "отложенный test используется только для измерения, гипотеза по нему не подбиралась.\n\n"
        f"**Наблюдения.** AUC: random={rows['random']['auc']}, entropy={rows['entropy']['auc']}, "
        f"entropy_diverse={rows['entropy_diverse']['auc']}±{float(np.std(auc_div)):.4f} "
        f"(по сидам: {auc_div}). Итоговая accuracy: {rows}.\n\n"
        f"**Вывод.** Гипотеза {'подтверждена' if imp > 0 else 'не подтверждена'} "
        f"(ΔAUC vs entropy = {imp:+.4f}). {expl}.", encoding="utf-8")
    print(f"Бонус: AUC random={rows['random']['auc']}, entropy={rows['entropy']['auc']}, "
          f"entropy_diverse={rows['entropy_diverse']['auc']} (Δ={imp:+.4f}) -> artifacts/bonus.md + bonus.json")
else:
    print("Бонус пропущен (RUN_BONUS=False)")

Бонус: AUC random=0.8795, entropy=0.8448, entropy_diverse=0.8624 (Δ=+0.0176) -> artifacts/bonus.md + bonus.json


## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.